# KAN-AD baseline on SMAP

This notebook adapts the uploaded KAN-AD SWaT baseline logic to SMAP.

In [1]:
# Cell 1 — Setup

!pip install --upgrade pip
!pip install toml torch torchinfo optuna
!pip install git+https://github.com/CSTCloudOps/EasyTSAD.git

!rm -rf KAN-AD datasets
!git clone https://github.com/CSTCloudOps/KAN-AD.git
!git clone https://github.com/CSTCloudOps/datasets.git
!mv datasets KAN-AD/datasets

%cd KAN-AD

import sys, os, random
import numpy as np
import torch

project_path = os.getcwd()
if project_path not in sys.path:
    sys.path.insert(0, project_path)

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

print("🟢 Paths configured.")
print("Current working directory:", project_path)

!sed -i 's/TSData、/TSData/g; s/TSData,*/TSData/g' \
    /usr/local/lib/python3.12/dist-packages/EasyTSAD/DataFactory/__init__.py

from EasyTSAD.Controller import TSADController
from EasyTSAD.DataFactory import TSData
from EasyTSAD.Methods import BaseMethod
print("✅ EasyTSAD ready")


  Cloning https://github.com/CSTCloudOps/EasyTSAD.git to /tmp/pip-req-build-r9lcf89b
  Running command git clone --filter=blob:none --quiet https://github.com/CSTCloudOps/EasyTSAD.git /tmp/pip-req-build-r9lcf89b
  Resolved https://github.com/CSTCloudOps/EasyTSAD.git to commit 507e5533a142eec7eece4372c55e83bb3faff8a1
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Cloning into 'KAN-AD'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 18 (delta 1), reused 15 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 61.77 KiB | 559.00 KiB/s, done.
Resolving deltas: 100% (1/1), done.
Cloning into 'datasets'...
remote: Enumerating objects: 4503, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 4503 (delta 2), reused 0 (delta 0), pack-r

In [2]:
# Cell 2 — Convert SMAP MTS AllInOne to channel-independent UTS

import os, json
import numpy as np

ORIG_DATASET = "SMAP"
CUSTOM_DATASET = "SMAP_CI"

src = f"/content/KAN-AD/datasets/MTS/{ORIG_DATASET}/AllInOne"
dst_root = f"/content/KAN-AD/datasets/UTS/{CUSTOM_DATASET}"

assert os.path.exists(src), f"Source dataset not found: {src}"

train = np.load(os.path.join(src, "train.npy"))
test  = np.load(os.path.join(src, "test.npy"))
train_label = np.load(os.path.join(src, "train_label.npy"))
test_label  = np.load(os.path.join(src, "test_label.npy"))

assert train.ndim == 2 and test.ndim == 2
Ttr, D = train.shape
Tte, D2 = test.shape
assert D == D2

def to_time_labels(y):
    y = np.asarray(y)
    if y.ndim == 1:
        return (y != 0).astype(np.int64)
    if y.ndim == 2:
        return (y.sum(axis=1) != 0).astype(np.int64)
    raise ValueError("Unsupported label shape")

train_y = to_time_labels(train_label)
test_y  = to_time_labels(test_label)

# KAN-AD-style preprocessing: first-order differencing + global z-score per channel
def preprocess(x):
    x = np.diff(x)
    x = (x - x.mean()) / (x.std() + 1e-8)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    return x.astype(np.float32)

os.makedirs(dst_root, exist_ok=True)

for j in range(D):
    curve = f"channel_{j:03d}"
    out = os.path.join(dst_root, curve)
    os.makedirs(out, exist_ok=True)

    tr = preprocess(train[:, j])
    te = preprocess(test[:, j])

    train_y_proc = train_y[1:]
    test_y_proc  = test_y[1:]

    tr_ts = np.arange(len(tr), dtype=np.int64)
    te_ts = np.arange(len(te), dtype=np.int64)

    np.save(os.path.join(out, "train.npy"), tr)
    np.save(os.path.join(out, "test.npy"), te)
    np.save(os.path.join(out, "train_label.npy"), train_y_proc)
    np.save(os.path.join(out, "test_label.npy"), test_y_proc)
    np.save(os.path.join(out, "train_timestamp.npy"), tr_ts)
    np.save(os.path.join(out, "test_timestamp.npy"), te_ts)

    info = {
        "dataset": CUSTOM_DATASET,
        "curve": curve,
        "type": "UTS",
        "length": int(len(tr) + len(te)),
        "train_length": int(len(tr)),
        "test_length": int(len(te)),
        "feature_dim": 1
    }

    with open(os.path.join(out, "info.json"), "w") as f:
        json.dump(info, f, indent=2)

print(f"✅ {ORIG_DATASET} converted to channel-independent UTS: {CUSTOM_DATASET}")
print("Original shape:", train.shape, test.shape)
print("Total channels:", D)
print("Example files:", os.listdir(os.path.join(dst_root, "channel_000")))


✅ SMAP converted to channel-independent UTS: SMAP_CI
Original shape: (135183, 25) (427617, 25)
Total channels: 25
Example files: ['test.npy', 'info.json', 'train_timestamp.npy', 'test_label.npy', 'train_label.npy', 'train.npy', 'test_timestamp.npy']


In [3]:
# Cell 3 — Train KAN-AD on SMAP_CI

import sys
sys.path.insert(0, "/content/KAN-AD")

from EasyTSAD.Controller import TSADController
from kanad import KANAD  # registers method

DATASET = "SMAP_CI"
METHOD_NAME = "KANAD"
TRAINING_SCHEMA = "naive"

gctrl = TSADController()

gctrl.set_dataset(
    dataset_type="UTS",
    dirname="datasets",
    datasets=[DATASET],
)

print(f"🚀 Starting KAN-AD training on {DATASET}...")

gctrl.run_exps(
    method=METHOD_NAME,
    training_schema=TRAINING_SCHEMA,
    cfg_path="kanad/config.toml",
)

print("✅ Training finished.")


(2026-06-05 22:50:43,935) [INFO]: 
                         
███████╗ █████╗ ███████╗██╗   ██╗    ████████╗███████╗ █████╗ ██████╗ 
██╔════╝██╔══██╗██╔════╝╚██╗ ██╔╝    ╚══██╔══╝██╔════╝██╔══██╗██╔══██╗
█████╗  ███████║███████╗ ╚████╔╝        ██║   ███████╗███████║██║  ██║
██╔══╝  ██╔══██║╚════██║  ╚██╔╝         ██║   ╚════██║██╔══██║██║  ██║
███████╗██║  ██║███████║   ██║          ██║   ███████║██║  ██║██████╔╝
╚══════╝╚═╝  ╚═╝╚══════╝   ╚═╝          ╚═╝   ╚══════╝╚═╝  ╚═╝╚═════╝ 
                                                                      
                         
INFO:logger:
                         
███████╗ █████╗ ███████╗██╗   ██╗    ████████╗███████╗ █████╗ ██████╗ 
██╔════╝██╔══██╗██╔════╝╚██╗ ██╔╝    ╚══██╔══╝██╔════╝██╔══██╗██╔══██╗
█████╗  ███████║███████╗ ╚████╔╝        ██║   ███████╗███████║██║  ██║
██╔══╝  ██╔══██║╚════██║  ╚██╔╝         ██║   ╚════██║██╔══██║██║  ██║
███████╗██║  ██║███████║   ██║          ██║   ███████║██║  ██║██████╔╝
╚══════╝╚═╝  ╚═╝╚═════

🚀 Starting KAN-AD training on SMAP_CI...


(2026-06-05 22:50:44,169) [INFO]:     [KANAD] handling dataset SMAP_CI | curve channel_008 
INFO:logger:    [KANAD] handling dataset SMAP_CI | curve channel_008 


=== Using CUDA ===


Validation Epoch [5/100]: 100%|██████████| 27/27 [00:00<00:00, 51.39it/s, avg_loss=0.0961, loss=3.05e-5]


...

EarlyStopping counter: 2 out of 3


Validation Epoch [10/100]: 100%|██████████| 27/27 [00:00<00:00, 88.84it/s, avg_loss=0.0843, loss=0.000131]


EarlyStopping counter: 3 out of 3
   Early stopping<<<


Testing: : 100%|██████████| 418/418 [00:04<00:00, 88.29it/s]


✅ Training finished.


In [4]:
# Cell 6 — Evaluate KAN-AD on SMAP_CI

from EasyTSAD.Evaluations.Protocols import (
    EventF1PA,
    PointF1PA,
    PointKthF1PA,
    PointAuprcPA,
)

method = METHOD_NAME
training_schema = TRAINING_SCHEMA

gctrl.set_evals([
    PointF1PA(),
    EventF1PA(mode="squeeze"),
    PointKthF1PA(k=5),
    PointAuprcPA(),
])

gctrl.do_evals(method=method, training_schema=training_schema)

print("✅ Evaluation completed")


(2026-06-05 23:03:04,826) [INFO]: Register evaluations
INFO:logger:Register evaluations
(2026-06-05 23:03:04,827) [INFO]: Perform evaluations. Method[KANAD], Schema[naive].
INFO:logger:Perform evaluations. Method[KANAD], Schema[naive].
(2026-06-05 23:03:04,829) [INFO]:     [Load Data (All)] DataSets: SMAP_CI 
INFO:logger:    [Load Data (All)] DataSets: SMAP_CI 
(2026-06-05 23:03:04,900) [INFO]:     [KANAD] Eval dataset SMAP_CI <<<
INFO:logger:    [KANAD] Eval dataset SMAP_CI <<<
(2026-06-05 23:03:04,903) [INFO]:         [SMAP_CI] Using default margins (0, 5)
INFO:logger:        [SMAP_CI] Using default margins (0, 5)


✅ Evaluation completed


In [5]:
# Cell 5 — Load KAN-AD results

import json, os

DATASET = "SMAP_CI"
METHOD_NAME = "KANAD"
TRAINING_SCHEMA = "naive"
DISPLAY_MODEL = "KAN-AD"

avg_path = f"/content/KAN-AD/Results/Evals/{METHOD_NAME}/{TRAINING_SCHEMA}/{DATASET}/avg.json"
all_path = f"/content/KAN-AD/Results/Evals/{METHOD_NAME}/{TRAINING_SCHEMA}/{DATASET}/all.json"

assert os.path.exists(avg_path), f"avg.json not found: {avg_path}"
assert os.path.exists(all_path), f"all.json not found: {all_path}"

with open(avg_path, "r") as f:
    avg = json.load(f)
with open(all_path, "r") as f:
    all_scores = json.load(f)

print("=== AVERAGE RESULTS (KAN-AD - SMAP_CI) ===")
for k, v in avg.items():
    print(f"{k}: {v}")

row = {
    "Model": DISPLAY_MODEL,
    "Dataset": "SMAP",
    "F1": avg["best f1 under pa"]["f1"],
    "Precision": avg["best f1 under pa"]["precision"],
    "Recall": avg["best f1 under pa"]["recall"],
    "AUPRC": avg["point-based auprc pa"],
}
print("\n=== Clean row for comparison table ===")
for k, v in row.items():
    print(f"{k}: {v}")


=== AVERAGE RESULTS (KAN-AD - SMAP_CI) ===
best f1 under pa: {'f1': 0.5703906898508141, 'precision': 0.9158521159796369, 'recall': 0.44906834329741424}
event-based f1 under pa with mode squeeze: {'f1': 0.16967314295429867, 'precision': 0.3053501540406853, 'recall': 0.14925373134328357}
best f1 under 5-delay pa: {'f1': 0.31912137701401627, 'precision': 0.23115526512532675, 'recall': 0.5426410568588614}
point-based auprc pa: 0.5014923121611702

=== Clean row for comparison table ===
Model: KAN-AD
Dataset: SMAP
F1: 0.5703906898508141
Precision: 0.9158521159796369
Recall: 0.44906834329741424
AUPRC: 0.5014923121611702
